# **# Spark Architecture**

Apache Spark consists of:

- **Driver Program** – Controls the Spark application and creates jobs.
- **SparkSession** – Entry point to Spark.
- **Cluster Manager** – Allocates resources.
- **Executors** – Execute tasks and process data.
- **Worker Nodes** – Machines that run executors.

**Architecture:**

```text
             Driver
                |
          SparkSession
                |
         Cluster Manager
                |
    -------------------------
    |           |           |
 Executor      Executor   Executor
```

# **# Spark Execution Modes**

1. Local Mode – Runs on a single computer.
2. Standalone Mode – Spark's built-in cluster manager.
3. YARN Mode – Runs on Hadoop clusters.
4. Kubernetes Mode – Runs Spark inside Kubernetes.

In [1]:
!apt-get install openjdk-11-jdk-headless -qq > /dev/null
!wget -q https://archive.apache.org/dist/spark/spark-3.5.1/spark-3.5.1-bin-hadoop3.tgz
!tar xf spark-3.5.1-bin-hadoop3.tgz
!pip install -q findspark

In [12]:
import os
import findspark
from pyspark.sql.functions import *

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-3.5.1-bin-hadoop3"

findspark.init()

In [3]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("SparkLab") \
    .getOrCreate()

print("Spark Started Successfully!")

Spark Started Successfully!


**Read the CSV File**

In [38]:
df = spark.read.csv(
    "sales_dataset.csv",
    header=True,
    inferSchema=True
)

df.show()

+-------+----------------+------+----------------+-----------+-----+---+------------+-----+--------+------------------+--------+---------+----------------+
|user_id|transaction_date|region|product_category|sale_amount| city|age|subscription|price|store_id|             email|username|   status|   raw_timestamp|
+-------+----------------+------+----------------+-----------+-----+---+------------+-----+--------+------------------+--------+---------+----------------+
|   1001|      01-01-2025|  West|     Electronics|        100|Delhi| 18|     Premium| NULL|    S101|              NULL|    NULL|     NULL|01-01-2025 10:00|
|   1002|      02-01-2025|  West|       Furniture|        105|Delhi| 19|       Basic|  105|    S101|user1002@gmail.com|user1002|Completed|02-01-2025 10:01|
|   1003|      03-01-2025|  West|        Clothing|        110|Delhi| 20|     Premium|  110|    S101|user1003@gmail.com|user1003|Completed|03-01-2025 10:02|
|   1004|      04-01-2025|  West|         Grocery|        115|De

In [39]:
df.printSchema()

root
 |-- user_id: integer (nullable = true)
 |-- transaction_date: string (nullable = true)
 |-- region: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- sale_amount: integer (nullable = true)
 |-- city: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- subscription: string (nullable = true)
 |-- price: integer (nullable = true)
 |-- store_id: string (nullable = true)
 |-- email: string (nullable = true)
 |-- username: string (nullable = true)
 |-- status: string (nullable = true)
 |-- raw_timestamp: string (nullable = true)



In [40]:
df.count()

401

In [42]:
# Q5: Given a DataFrame df, write a query to select the columns product_id and price where the category is 'Electronics'.
from pyspark.sql.functions import col

selected_df = df.filter(col("product_category") == "Electronics") \
                .select("user_id", "price")

selected_df.show()

+-------+-----+
|user_id|price|
+-------+-----+
|   1001| NULL|
|   1005|  120|
|   1009|  140|
|   1013|  160|
|   1017|  180|
|   1021|  200|
|   1025|  220|
|   1029|  240|
|   1033|  260|
|   1037|  280|
|   1041|  300|
|   1045|  320|
|   1049|  340|
|   1053|  360|
|   1057|  380|
|   1061|  400|
|   1065|  420|
|   1069| NULL|
|   1073|  460|
|   1077|  480|
+-------+-----+
only showing top 20 rows



# **# Lazy Evaluation**

Spark does not execute transformations immediately.

Transformation Example:
filter(), select(), withColumn()

Execution starts only when an Action such as show(), count(), or write() is called.

Spark creates a DAG (Directed Acyclic Graph) and optimizes execution before running.

### **Step 1:** Filter age Older Than 30

In [43]:
age_df = df.filter(col("age") > 30)
age_df.show()

+-------+----------------+------+----------------+-----------+-----+---+------------+-----+--------+------------------+--------+---------+----------------+
|user_id|transaction_date|region|product_category|sale_amount| city|age|subscription|price|store_id|             email|username|   status|   raw_timestamp|
+-------+----------------+------+----------------+-----------+-----+---+------------+-----+--------+------------------+--------+---------+----------------+
|   1014|      14-01-2025|  West|       Furniture|        165|Delhi| 31|       Basic|  165|    S101|user1014@gmail.com|user1014|Completed|14-01-2025 10:13|
|   1015|      15-01-2025|  West|        Clothing|        170|Delhi| 32|     Premium|  170|    S101|user1015@gmail.com|user1015|Completed|15-01-2025 10:14|
|   1016|      16-01-2025|  West|         Grocery|        175|Delhi| 33|       Basic|  175|    S101|user1016@gmail.com|user1016|Completed|16-01-2025 10:15|
|   1017|      17-01-2025|  West|     Electronics|        180|De

### **Step 2:** Filter Completed Transactions

In [44]:
leave_df = df.filter(col("status") == "Completed")

leave_df.show()

+-------+----------------+------+----------------+-----------+-----+---+------------+-----+--------+------------------+--------+---------+----------------+
|user_id|transaction_date|region|product_category|sale_amount| city|age|subscription|price|store_id|             email|username|   status|   raw_timestamp|
+-------+----------------+------+----------------+-----------+-----+---+------------+-----+--------+------------------+--------+---------+----------------+
|   1002|      02-01-2025|  West|       Furniture|        105|Delhi| 19|       Basic|  105|    S101|user1002@gmail.com|user1002|Completed|02-01-2025 10:01|
|   1003|      03-01-2025|  West|        Clothing|        110|Delhi| 20|     Premium|  110|    S101|user1003@gmail.com|user1003|Completed|03-01-2025 10:02|
|   1004|      04-01-2025|  West|         Grocery|        115|Delhi| 21|       Basic|  115|    S101|user1004@gmail.com|user1004|Completed|04-01-2025 10:03|
|   1005|      05-01-2025|  West|     Electronics|        120|De

### **Step 3:** Filter Multiple Conditions

In [67]:
df.filter(
    (col("age") > 30) &
    (col("subscription") == "Premium")
).show()

+-------+----------------+------+----------------+----------+------+----+------------+-----+--------+------------------+--------+---------+----------------+------------------+-----------------+
|user_id|transaction_date|region|product_category|total_sale|  city| age|subscription|price|store_id|             email|username|   status|   raw_timestamp|       final_price|TransactionStatus|
+-------+----------------+------+----------------+----------+------+----+------------+-----+--------+------------------+--------+---------+----------------+------------------+-----------------+
|   1015|      15-01-2025|  West|        Clothing|       170| Delhi|32.0|     Premium|  170|    S101|user1015@gmail.com|user1015|Completed|15-01-2025 10:14|187.00000000000003|          Success|
|   1017|      17-01-2025|  West|     Electronics|       180| Delhi|34.0|     Premium|  180|    S101|user1017@gmail.com|user1017|Completed|17-01-2025 10:16|198.00000000000003|          Success|
|   1033|      05-01-2025|  We

### **Step 4:** Rename a Column - ques 6

In [46]:
df = df.withColumnRenamed(
    "sale_amount",
    "total_sale"
)

df.show(5)

+-------+----------------+------+----------------+----------+-----+---+------------+-----+--------+------------------+--------+---------+----------------+
|user_id|transaction_date|region|product_category|total_sale| city|age|subscription|price|store_id|             email|username|   status|   raw_timestamp|
+-------+----------------+------+----------------+----------+-----+---+------------+-----+--------+------------------+--------+---------+----------------+
|   1001|      01-01-2025|  West|     Electronics|       100|Delhi| 18|     Premium| NULL|    S101|              NULL|    NULL|     NULL|01-01-2025 10:00|
|   1002|      02-01-2025|  West|       Furniture|       105|Delhi| 19|       Basic|  105|    S101|user1002@gmail.com|user1002|Completed|02-01-2025 10:01|
|   1003|      03-01-2025|  West|        Clothing|       110|Delhi| 20|     Premium|  110|    S101|user1003@gmail.com|user1003|Completed|03-01-2025 10:02|
|   1004|      04-01-2025|  West|         Grocery|       115|Delhi| 21

### **Step 5:** Cast Data Type

In [70]:
df = df.withColumn(
    "age",
    col("age").cast("double")
)

df.show()

+-------+----------------+------+----------------+----------+-----+----+------------+-----+--------+------------------+--------+---------+----------------+------------------+-----------------+
|user_id|transaction_date|region|product_category|total_sale| city| age|subscription|price|store_id|             email|username|   status|   raw_timestamp|       final_price|TransactionStatus|
+-------+----------------+------+----------------+----------+-----+----+------------+-----+--------+------------------+--------+---------+----------------+------------------+-----------------+
|   1001|      01-01-2025|  West|     Electronics|       100|Delhi|18.0|     Premium| NULL|    S101|              NULL|    NULL|     NULL|01-01-2025 10:00|              NULL|          Unknown|
|   1002|      02-01-2025|  West|       Furniture|       105|Delhi|19.0|       Basic|  105|    S101|user1002@gmail.com|user1002|Completed|02-01-2025 10:01|115.50000000000001|          Success|
|   1003|      03-01-2025|  West|  

Q10: Write a code snippet to add a new column final_price which is the base_price multiplied by 1.18 (18% tax).

In [72]:
from pyspark.sql.functions import col
df = df.withColumn(
    "final_price",
    col("price") * 1.18
)
df.show(5)


+-------+----------------+------+----------------+----------+-----+----+------------+-----+--------+------------------+--------+---------+----------------+------------------+-----------------+
|user_id|transaction_date|region|product_category|total_sale| city| age|subscription|price|store_id|             email|username|   status|   raw_timestamp|       final_price|TransactionStatus|
+-------+----------------+------+----------------+----------+-----+----+------------+-----+--------+------------------+--------+---------+----------------+------------------+-----------------+
|   1001|      01-01-2025|  West|     Electronics|       100|Delhi|18.0|     Premium| NULL|    S101|              NULL|    NULL|     NULL|01-01-2025 10:00|              NULL|          Unknown|
|   1002|      02-01-2025|  West|       Furniture|       105|Delhi|19.0|       Basic|  105|    S101|user1002@gmail.com|user1002|Completed|02-01-2025 10:01|123.89999999999999|          Success|
|   1003|      03-01-2025|  West|  

### **Step 6:** Add a New Column

In [48]:
df = df.withColumn(
    "final_price",
    col("price") * 1.10
)

df.show(5)

+-------+----------------+------+----------------+----------+-----+----+------------+-----+--------+------------------+--------+---------+----------------+------------------+
|user_id|transaction_date|region|product_category|total_sale| city| age|subscription|price|store_id|             email|username|   status|   raw_timestamp|       final_price|
+-------+----------------+------+----------------+----------+-----+----+------------+-----+--------+------------------+--------+---------+----------------+------------------+
|   1001|      01-01-2025|  West|     Electronics|       100|Delhi|18.0|     Premium| NULL|    S101|              NULL|    NULL|     NULL|01-01-2025 10:00|              NULL|
|   1002|      02-01-2025|  West|       Furniture|       105|Delhi|19.0|       Basic|  105|    S101|user1002@gmail.com|user1002|Completed|02-01-2025 10:01|115.50000000000001|
|   1003|      03-01-2025|  West|        Clothing|       110|Delhi|20.0|     Premium|  110|    S101|user1003@gmail.com|user10

### **Step 7:** Add Conditional Column

In [49]:
df = df.withColumn(
    "TransactionStatus",
    when(col("status") == "Completed", "Success")
    .when(col("status") == "Pending", "InProgress")
    .otherwise("Unknown")
)

df.show(5)

+-------+----------------+------+----------------+----------+-----+----+------------+-----+--------+------------------+--------+---------+----------------+------------------+-----------------+
|user_id|transaction_date|region|product_category|total_sale| city| age|subscription|price|store_id|             email|username|   status|   raw_timestamp|       final_price|TransactionStatus|
+-------+----------------+------+----------------+----------+-----+----+------------+-----+--------+------------------+--------+---------+----------------+------------------+-----------------+
|   1001|      01-01-2025|  West|     Electronics|       100|Delhi|18.0|     Premium| NULL|    S101|              NULL|    NULL|     NULL|01-01-2025 10:00|              NULL|          Unknown|
|   1002|      02-01-2025|  West|       Furniture|       105|Delhi|19.0|       Basic|  105|    S101|user1002@gmail.com|user1002|Completed|02-01-2025 10:01|115.50000000000001|          Success|
|   1003|      03-01-2025|  West|  

### **Step 8:** Check Null Values

In [50]:
from pyspark.sql.functions import isnan

df.select([
    col(c).isNull().alias(c)
    for c in df.columns
]).show()

+-------+----------------+------+----------------+----------+-----+-----+------------+-----+--------+-----+--------+------+-------------+-----------+-----------------+
|user_id|transaction_date|region|product_category|total_sale| city|  age|subscription|price|store_id|email|username|status|raw_timestamp|final_price|TransactionStatus|
+-------+----------------+------+----------------+----------+-----+-----+------------+-----+--------+-----+--------+------+-------------+-----------+-----------------+
|  false|           false| false|           false|     false|false|false|       false| true|   false| true|    true|  true|        false|       true|            false|
|  false|           false| false|           false|     false|false|false|       false|false|   false|false|   false| false|        false|      false|            false|
|  false|           false| false|           false|     false|false|false|       false|false|   false|false|   false| false|        false|      false|           

### **Step 9:** Remove Null Rows

In [51]:
clean_df = df.dropna()

clean_df.show()

+-------+----------------+------+----------------+----------+-----+----+------------+-----+--------+------------------+--------+---------+----------------+------------------+-----------------+
|user_id|transaction_date|region|product_category|total_sale| city| age|subscription|price|store_id|             email|username|   status|   raw_timestamp|       final_price|TransactionStatus|
+-------+----------------+------+----------------+----------+-----+----+------------+-----+--------+------------------+--------+---------+----------------+------------------+-----------------+
|   1002|      02-01-2025|  West|       Furniture|       105|Delhi|19.0|       Basic|  105|    S101|user1002@gmail.com|user1002|Completed|02-01-2025 10:01|115.50000000000001|          Success|
|   1003|      03-01-2025|  West|        Clothing|       110|Delhi|20.0|     Premium|  110|    S101|user1003@gmail.com|user1003|Completed|03-01-2025 10:02|121.00000000000001|          Success|
|   1004|      04-01-2025|  West|  

### **Step 10**: Fill Null Values

In [52]:
filled_df = df.fillna({
    "city":"Unknown"
})

filled_df.show()

+-------+----------------+------+----------------+----------+-----+----+------------+-----+--------+------------------+--------+---------+----------------+------------------+-----------------+
|user_id|transaction_date|region|product_category|total_sale| city| age|subscription|price|store_id|             email|username|   status|   raw_timestamp|       final_price|TransactionStatus|
+-------+----------------+------+----------------+----------+-----+----+------------+-----+--------+------------------+--------+---------+----------------+------------------+-----------------+
|   1001|      01-01-2025|  West|     Electronics|       100|Delhi|18.0|     Premium| NULL|    S101|              NULL|    NULL|     NULL|01-01-2025 10:00|              NULL|          Unknown|
|   1002|      02-01-2025|  West|       Furniture|       105|Delhi|19.0|       Basic|  105|    S101|user1002@gmail.com|user1002|Completed|02-01-2025 10:01|115.50000000000001|          Success|
|   1003|      03-01-2025|  West|  

# **# Wide Transformation (Shuffle)**

groupBy() is a Wide Transformation.

Spark redistributes data across executors.
This redistribution is called Shuffle.

Shuffle increases network communication and is more expensive than narrow transformations.

### **Step 11:** Wide Transformation (Shuffle)

In [53]:
city_count = df.groupBy("city").count()

city_count.show()

+-------+-----+
|   city|count|
+-------+-----+
|Chennai|   60|
| Mumbai|  110|
|Kolkata|  110|
|  Delhi|  121|
+-------+-----+



### **Step 12:** Another Wide Transformation

In [54]:
df.groupBy("city").avg("age").show()

+-------+------------------+
|   city|          avg(age)|
+-------+------------------+
|Chennai|              28.5|
| Mumbai|26.772727272727273|
|Kolkata|26.772727272727273|
|  Delhi|26.132231404958677|
+-------+------------------+



### **Step 13:** Explain Execution Plan (DAG)

In [55]:
city_count.explain(True)

== Parsed Logical Plan ==
'Aggregate ['city], ['city, count(1) AS count#2261L]
+- Project [user_id#1292, transaction_date#1293, region#1294, product_category#1295, total_sale#1638, city#1297, age#1724, subscription#1299, price#1300, store_id#1301, email#1302, username#1303, status#1304, raw_timestamp#1305, final_price#1739, CASE WHEN (status#1304 = Completed) THEN Success WHEN (status#1304 = Pending) THEN InProgress ELSE Unknown END AS TransactionStatus#1830]
   +- Project [user_id#1292, transaction_date#1293, region#1294, product_category#1295, total_sale#1638, city#1297, age#1724, subscription#1299, price#1300, store_id#1301, email#1302, username#1303, status#1304, raw_timestamp#1305, (cast(price#1300 as double) * 1.1) AS final_price#1739]
      +- Project [user_id#1292, transaction_date#1293, region#1294, product_category#1295, total_sale#1638, city#1297, cast(age#1298 as double) AS age#1724, subscription#1299, price#1300, store_id#1301, email#1302, username#1303, status#1304, raw_t

### **Step 14:** Save as CSV

In [56]:
clean_df.write \
.mode("overwrite") \
.option("header",True) \
.csv("Sales_Output_CSV")

### **Step 15:** Save as Parquet

In [57]:
clean_df.write \
.mode("overwrite") \
.parquet("Sales_Output_Parquet")

### **Step 16:** Read Parquet Back

In [58]:
parquet_df = spark.read.parquet(
    "Sales_Output_Parquet"
)

parquet_df.show(5)

+-------+----------------+------+----------------+----------+-----+----+------------+-----+--------+------------------+--------+---------+----------------+------------------+-----------------+
|user_id|transaction_date|region|product_category|total_sale| city| age|subscription|price|store_id|             email|username|   status|   raw_timestamp|       final_price|TransactionStatus|
+-------+----------------+------+----------------+----------+-----+----+------------+-----+--------+------------------+--------+---------+----------------+------------------+-----------------+
|   1002|      02-01-2025|  West|       Furniture|       105|Delhi|19.0|       Basic|  105|    S101|user1002@gmail.com|user1002|Completed|02-01-2025 10:01|115.50000000000001|          Success|
|   1003|      03-01-2025|  West|        Clothing|       110|Delhi|20.0|     Premium|  110|    S101|user1003@gmail.com|user1003|Completed|03-01-2025 10:02|121.00000000000001|          Success|
|   1004|      04-01-2025|  West|  

# **# Predicate Pushdown**

Parquet supports Predicate Pushdown.

When filters are applied, Spark reads only the required rows instead of scanning the complete file.

This reduces disk I/O and improves performance.

## **Step 17:** Compare CSV vs Parquet

| CSV                   | Parquet                     |
| --------------------- | --------------------------- |
| Text format           | Columnar format             |
| Larger size           | Smaller size                |
| Slower                | Faster                      |
| No schema stored      | Schema stored               |
| No Predicate Pushdown | Supports Predicate Pushdown |


### **Step 18:** Complete Data Pipeline

In [62]:
from pyspark.sql.functions import col

pipeline_df = (
    df
    .filter(col("age") > 25)
    .select(
        "user_id",
        "product_category",
        "city",
        "age",
        "total_sale",
        "status"
    )
    .withColumn("AgeAfter5Years", col("age") + 5)
    .withColumn("HighValueSale", col("total_sale") > 300)
)

pipeline_df.show()

pipeline_df.write.mode("overwrite").parquet("Final_Output")

+-------+----------------+-----+----+----------+---------+--------------+-------------+
|user_id|product_category| city| age|total_sale|   status|AgeAfter5Years|HighValueSale|
+-------+----------------+-----+----+----------+---------+--------------+-------------+
|   1009|     Electronics|Delhi|26.0|       140|Completed|          31.0|        false|
|   1010|       Furniture|Delhi|27.0|       145|Completed|          32.0|        false|
|   1011|        Clothing|Delhi|28.0|       150|Completed|          33.0|        false|
|   1012|         Grocery|Delhi|29.0|       155|Completed|          34.0|        false|
|   1013|     Electronics|Delhi|30.0|       160|Completed|          35.0|        false|
|   1014|       Furniture|Delhi|31.0|       165|Completed|          36.0|        false|
|   1015|        Clothing|Delhi|32.0|       170|Completed|          37.0|        false|
|   1016|         Grocery|Delhi|33.0|       175|Completed|          38.0|        false|
|   1017|     Electronics|Delhi|

In [61]:
pipeline_df.explain(True)

== Parsed Logical Plan ==
'Project [user_id#1292, product_category#1295, city#1297, age#1724, total_sale#1638, status#1304, AgeAfter5Years#2479, ('total_sale > 300) AS HighValueSale#2487]
+- Project [user_id#1292, product_category#1295, city#1297, age#1724, total_sale#1638, status#1304, (age#1724 + cast(5 as double)) AS AgeAfter5Years#2479]
   +- Project [user_id#1292, product_category#1295, city#1297, age#1724, total_sale#1638, status#1304]
      +- Filter (age#1724 > cast(25 as double))
         +- Project [user_id#1292, transaction_date#1293, region#1294, product_category#1295, total_sale#1638, city#1297, age#1724, subscription#1299, price#1300, store_id#1301, email#1302, username#1303, status#1304, raw_timestamp#1305, final_price#1739, CASE WHEN (status#1304 = Completed) THEN Success WHEN (status#1304 = Pending) THEN InProgress ELSE Unknown END AS TransactionStatus#1830]
            +- Project [user_id#1292, transaction_date#1293, region#1294, product_category#1295, total_sale#1638

# **Performance Insights**

- Spark uses Lazy Evaluation to optimize execution.
- DAG optimization reduces unnecessary computations.
- `groupBy()` is a wide transformation and causes Shuffle.
- Parquet performs better than CSV because it is columnar.
- Predicate Pushdown reduces data scanning.
- `show()` is preferred over `collect()` for large datasets.